# 자동 이미지 캡션 생성하기

In [1]:
from dotenv import load_dotenv
import os

# .env 파일의 내용 불러오기
load_dotenv("/Users/leelaal/.env")

# 환경 변수 가져오기
API_KEY = os.getenv("OPENAI_API_KEY")
# print(API_KEY)

from openai import OpenAI
client = OpenAI(api_key=API_KEY)

from flask import Flask, request, jsonify, render_template
app = Flask(__name__)

### 이미지 캡션만 출력하기

In [2]:
import cv2
import numpy as np
import base64

app = Flask(__name__)

# 이미지를 Base64로 인코딩
def encode_image_to_base64(image):    
    _, buffer = cv2.imencode('.jpg', image)
    return base64.b64encode(buffer).decode('utf-8')

# OpenAI API를 사용하여 이미지 캡션 생성
def generate_caption(image_base64):
    response = client.chat.completions.create(
      model = "gpt-4o-mini",     
      messages = [
        {
          "role": "user",
          "content": [
            {"type": "text", "text": "이미지의 내용에대한 캡션을 짧게 달아줘"},
            {
              "type": "image_url",
              "image_url": {
                 "url": f"data:image/jpeg;base64,{image_base64}"
              },
            },
          ],
        }
      ],
      max_tokens=100,
    )    
    return response.choices[0].message.content

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload():
    file = request.files['file']   # 업로드 된 파일을 받아온다
    if not file:
        return jsonify({'error': '파일을 찾을 수 없습니다.'}), 400

    # OpenCV로 이미지 읽기
    npimg = np.frombuffer(file.read(), np.uint8)  # 바이너리(raw byte stream)로 읽은 후 NumPy 배열로 변환하여 반환
    image = cv2.imdecode(npimg, cv2.IMREAD_COLOR) # NumPy 배열을 3차원(H,W,C)구조의 컬러 이미지(BGR 포맷)으로 디코딩 변환
    # cv2.imread()는 파일 경로에서 이미지를 읽지만, cv2.imdecode()는 메모리에 있는 이미지 데이터를 직접 변환할 때 사용.

    # 이미지 Base64 인코딩
    image_base64 = encode_image_to_base64(image)

    # 캡션 생성
    caption = generate_caption(image_base64)

    return jsonify({'caption': caption})

if __name__ == '__main__':
    app.run(debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/May/2026 21:42:00] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 21:42:01] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:42:04] "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:42:04] "GET /apple-touch-icon.png HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:42:04] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:43:24] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 21:43:27] "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:43:27] "GET /apple-touch-icon.png HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:43:27] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [21/May/2026 21:45:12] "POST /upload HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 21:46:11] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 21:47:39] "GET / HTTP/1.1" 200 -


### 이미지 분석 및 특징 추출

In [ ]:

app = Flask(__name__)
UPLOAD_FOLDER = 'static/uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER


# 이미지를 Base64로 인코딩
def encode_image_to_base64(image):
    
    _, buffer = cv2.imencode('.jpg', image)
    return base64.b64encode(buffer).decode('utf-8')

# OpenAI API를 사용하여 이미지 설명 생성
def generate_caption(image_base64):
    response = client.chat.completions.create(
      model = "gpt-4o-mini",     
      messages = [
        {
          "role": "user",
          "content": [
            {"type": "text", "text": "이미지의 내용에대한 캡션을 짧게 달아주고 주요 특징들을 자세히 설명해주세요"},
            {
              "type": "image_url",
              "image_url": {
                 "url": f"data:image/jpeg;base64,{image_base64}"
              },
            },
          ],
        }
      ],
      # max_tokens=100,
    )    
    return response.choices[0].message.content
    

# 이미지의 기본적인 전처리를 수행 
def process_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    return image, gray, edges

# 이미지에서 특징을 추출 
def extract_features(image):
    orb = cv2.ORB_create()
    keypoints, descriptors = orb.detectAndCompute(image, None)
    return len(keypoints)

def generate_final_caption(caption,feature_count):
    """ 추출된 특징을 기반으로 GPT를 활용하여 캡션을 생성 """
    prompt = f" {caption}이 있는 이 이미지에는 약 {feature_count}개의 주요 특징이 감지되었습니다. 주요 특징들을 다시 자세히 설명해주세요."
    # prompt = f" {caption}이 있는 이 이미지에는 약 여러 개의 주요 특징이 감지되었습니다. 주요 특징들을 자세히 설명해주세요."
    
    # OpenAI 모델에 요청
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages= [ {"role": "system", "content": "당신은 이미지 분석 AI 전문가입니다."},
                    {"role": "user", "content": prompt}] ,
        # max_tokens = 300,
        # n = 1,
        # stop = None,
        # temperature = 0.7,
    )
    
    return response.choices[0].message.content

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/upload', methods=['POST'])
def upload():
    file = request.files['file']   # 업로드 된 파일을 받아온다
    if not file:
        return jsonify({'error': '파일을 찾을 수 없습니다.'}), 400

    # OpenCV로 이미지 읽기
    npimg = np.frombuffer(file.read(), np.uint8)  # 바이너리(raw byte stream)로 읽은 후 NumPy 배열로 변환하여 반환
    image = cv2.imdecode(npimg, cv2.IMREAD_COLOR) # NumPy 배열을 3차원(H,W,C)구조의 컬러 이미지(BGR 포맷)으로 디코딩 변환
    # cv2.imread()는 파일 경로에서 이미지를 읽지만, cv2.imdecode()는 메모리에 있는 이미지 데이터를 직접 변환할 때 사용.

    # 이미지 Base64 인코딩
    image_base64 = encode_image_to_base64(image)

    # 캡션 생성
    caption = generate_caption(image_base64)

    _, gray, edges = process_image(image)
    feature_count = extract_features(gray)
    print(feature_count)
    caption2 = generate_final_caption(caption,feature_count)
    
        
    return jsonify({'caption': caption + "\n\n" + f'feauture count : {feature_count} ' + "\n" + caption2})

if __name__ == '__main__':
    app.run(debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [21/May/2026 16:20:52] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 16:20:56] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 16:21:11] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 16:21:12] "GET /favicon.ico HTTP/1.1" 404 -


500


127.0.0.1 - - [21/May/2026 16:21:49] "POST /upload HTTP/1.1" 200 -
127.0.0.1 - - [21/May/2026 16:22:09] "GET / HTTP/1.1" 200 -
